# [TIL] FastAPI 기업·채용공고 검색 API 개발 — CI 통과가 끝이 아니었다

**2026.11.09 (월) | KANT AX 부트캠프**

**오늘의 키워드:** FastAPI, SQLAlchemy 2, PostgreSQL, REST API, Keyset Pagination, Code Review, pytest, Claude Code

## 1. 오늘의 목표

오늘은 팀 프로젝트로 진행 중인 **AI 커리어 플랫폼**의 백엔드 개발을 담당했다.

내가 맡은 작업은 **T18 기업·채용공고 조회 및 검색 API 구현**이다.

기업과 채용공고를 조회하고, 검색 조건에 따라 필터링하며, 커서 기반 페이지네이션을 제공하는 기능이다.

기술 스택은 다음과 같다.

- Backend: FastAPI, SQLAlchemy 2
- Database: PostgreSQL 18
- Test: pytest, psycopg
- 협업: GitHub, Redmine
- AI 개발 도구: Claude, Claude Code
- 개발 환경: Windows 11, VS Code, PowerShell

오늘은 Claude와 Claude Code를 적극적으로 활용했다.

반복적인 코드 작성과 테스트 실행은 AI에게 맡기고, 나는 **작업 범위 확인, 설계 방향 결정, 오류 분석, 결과 검증**에 집중했다.

코드를 얼마나 많이 직접 작성했는지보다, 만들어진 코드가 요구사항을 충족하고 실제 환경에서도 올바르게 동작하는지를 판단하는 것이 중요했던 하루였다.

---

## 2. 개발 전, 프로젝트 상태부터 확인하기

바로 개발을 시작하지 않았다.

먼저 GitHub PR 목록과 Redmine 일감을 비교하면서 선행 작업이 어느 정도 완료됐는지 확인했다.

ERD, API 계약, 데이터베이스 스키마, Seed 데이터, FastAPI 기본 구조, DB 세션 등 T18 개발에 필요한 선행 작업은 모두 main 브랜치에 병합된 상태였다.

그런데 내 작업 브랜치는 커밋이 없었고, main보다 무려 83커밋이나 뒤처져 있었다.

그래서 최신 main을 기준으로 `task/t18-company-job-search-api` 브랜치를 다시 생성했다.

또 다른 팀원이 개발 중이던 직무·기술 API와 파일명이 충돌할 가능성을 발견했다.

이를 피하기 위해 `catalog.py` 대신 `job_postings.py`라는 파일명을 사용하기로 결정했다.

**배운 점:** 팀 프로젝트에서는 코드를 작성하기 전에 브랜치 상태와 다른 팀원의 작업 범위를 확인하는 과정도 개발의 일부다.

## 3. 기업·채용공고 API 구현

기존 `openapi.yaml`에 정의된 API 계약을 기준으로 네 가지 엔드포인트를 구현했다.

| Method | Endpoint | 기능 |
|---|---|---|
| GET | `/api/v1/companies` | 기업 목록 및 이름 검색 |
| GET | `/api/v1/companies/{id}` | 기업 상세 조회 |
| GET | `/api/v1/job-postings` | 채용공고 검색 및 필터링 |
| GET | `/api/v1/job-postings/{id}` | 채용공고 상세 조회 |

기업 목록에서는 이름 검색과 커서 페이지네이션을 지원한다.

채용공고 목록에서는 검색어, 기업, 직무를 기준으로 필터링할 수 있고, 마감 공고 포함 여부와 정렬 순서를 선택할 수 있도록 했다.

단순한 CRUD처럼 보였지만, 실제 구현 과정에서는 여러 가지 비즈니스 규칙을 고려해야 했다.

### 채용공고 공개 상태 판단

현재 게시 중인 채용공고는 다음 두 조건을 만족해야 한다.

1. `status = 'published'`
2. 마감일이 없거나 현재 시각보다 미래인 공고

반대로 마감일이 지난 published 공고는 응답에서 `closed`로 표현한다.

아직 공개되지 않은 `draft` 공고는 목록에서 제외하며, 상세 조회를 요청해도 404를 반환한다.

### 검색 및 필터링

검색어는 대소문자를 구분하지 않도록 처리했다.

특히 SQL LIKE 검색에서 `%`와 `_`는 특별한 의미를 갖기 때문에, 사용자가 입력한 문자를 그대로 검색할 수 있도록 이스케이프 처리를 적용했다.

또한 허용하지 않은 정렬값을 전달하면 임의로 처리하는 대신 API 계약에 맞춰 422 오류를 반환하도록 했다.

### 요구 기술 수준 기본값

채용공고에 요구 기술 수준이 명시돼 있지 않으면 해당 직무의 기본 요구 기술 정보인 `job_role_skills`를 사용하도록 했다.

다만 공고와 직무 기본값 양쪽에 모두 기술 수준이 없는 경우에는 `null`이 반환되는 문제가 남았다.

API 계약에서는 정수를 요구하므로, 이 부분은 팀 차원에서 추가 논의가 필요하다.

---

## 4. 첫 번째 실수 — 200 OK를 믿으면 안 된다

구현을 마친 뒤 로컬 서버를 실행해 API를 검증했다.

최소 권한으로 설정된 PostgreSQL 계정을 이용해 서버를 구동하고, curl 요청으로 응답을 확인하려 했다.

그런데 이상한 상황이 발생했다.

새로 실행한 서버에서는 오류가 발생했는데 API 요청은 모두 200 OK를 반환하는 것이었다.

로그를 확인하자 원인이 드러났다.

`ERROR: [Errno 98] address already in use`

이미 같은 포트에서 다른 uvicorn 프로세스가 실행되고 있었다.

기존 서버는 슈퍼유저 계정으로 실행 중이었고, 내가 새로 실행한 최소 권한 서버는 포트 충돌로 종료된 상태였다.

즉, curl 요청이 성공했다고 생각했지만 **실제로는 검증하려던 서버가 아닌 기존 서버가 응답하고 있었던 것이다.**

기존 프로세스를 종료한 뒤 서버를 다시 실행했다.

이후 PostgreSQL의 `pg_stat_activity`를 확인해 최소 권한 계정의 DB 연결이 실제로 생성되는지도 검증했다.

이 과정에서 정상 응답 9건과 예외 응답 6건을 확인했다.

**배운 점:** HTTP 200이라는 응답 코드만으로 검증이 끝난 것은 아니다. 어떤 프로세스가 요청을 처리했고 어떤 계정으로 DB에 접근했는지까지 확인해야 한다.

---

## 5. 코드 리뷰에서 발견한 Keyset Pagination 문제

오늘 가장 많은 것을 배운 부분은 **커서 기반 페이지네이션(Keyset Pagination)** 이었다.

처음 구현에서는 커서에 마지막으로 조회한 항목의 ID만 저장했다.

다음 페이지를 요청하면 그 ID를 이용해 DB에서 기준 행을 다시 가져오고, 그 행의 정렬값을 바탕으로 다음 데이터를 검색하는 구조였다.

처음에는 문제없이 동작하는 것처럼 보였다.

하지만 코드 리뷰 과정에서 이 구조가 데이터 변경에 취약하다는 사실을 발견했다.

### 문제 1. PostgreSQL infinity와 Python datetime 충돌

마감일이 없는 공고를 정렬하기 위해 PostgreSQL의 `COALESCE`와 `infinity`를 사용했다.

그런데 DB에서 infinity 값을 Python으로 가져오는 과정에서 오류가 발생했다.

`DataError: timestamp too large (after year 10K): 'infinity'`

PostgreSQL은 무한대 날짜를 표현할 수 있지만 Python의 `datetime`은 이를 정상적인 날짜 객체로 표현할 수 없었다.

해결 방법은 명확했다.

- 무한대 값은 SQL 내부에서만 처리한다.
- Python 커서에는 NULL을 포함한 실제 정렬 키를 저장한다.
- 다음 페이지 조회 시 기준 행을 다시 읽지 않는다.

### 문제 2. 데이터가 변경되면 페이지네이션이 깨진다

기존 구현에서는 마지막으로 조회한 공고가 다음 요청 전에 마감되면 400 오류가 발생할 수 있었다.

기업명으로 정렬하는 경우에도 기업명이 변경되면 다음 페이지가 비어 버리는 문제가 있었다.

원인은 동일했다.

**페이지 위치를 계산할 때 과거의 정렬값이 아니라 DB에서 다시 조회한 현재 값을 사용했기 때문이다.**

이를 해결하기 위해 커서 구조를 변경했다.

새로운 커서 v2에는 다음 정보가 포함된다.

`[버전, 목록 종류, 검색 조건 digest, 마지막 ID, 마지막 정렬 키]`

여기에 HMAC 서명을 적용해 클라이언트가 커서 값을 임의로 변경할 수 없도록 했다.

결과적으로 기준 행의 상태가 변하더라도 커서에 보관된 정렬값을 이용해 다음 페이지를 이어서 조회할 수 있게 됐다.

단, 이 방식이 모든 동시 데이터 변경 상황에서 완전한 스냅샷을 제공한다는 뜻은 아니다.

### 문제 3. 빈 문자열과 None은 다르다

또 하나의 작은 실수가 있었다.

기존 조건은 다음과 같았다.

`if cursor:`

하지만 Python에서 빈 문자열 `""`은 False다.

따라서 클라이언트가 `cursor=`처럼 빈 커서를 전달하면 커서가 없는 첫 페이지 요청으로 처리됐다.

이를 다음과 같이 변경했다.

`if cursor is not None:`

직접 확인한 결과는 다음과 같다.

- `bool("")` → `False`
- `"" is not None` → `True`

작은 조건문 하나의 차이였지만 API 검증 결과에 직접적인 영향을 줬다.

**배운 점:** 커서 기반 페이지네이션은 단순히 마지막 ID를 전달하는 기능이 아니다. 정렬 기준과 데이터 변경 가능성, NULL 처리, 커서 무결성까지 고려해야 한다.

---

## 6. 공개 API의 선택적 인증 처리

채용공고 상세 API에는 로그인 상태일 때 북마크 여부를 알려주는 `is_bookmarked` 필드가 있다.

로그인하지 않은 사용자도 공고를 조회할 수 있기 때문에 선택적 인증을 사용했다.

문제는 만료된 토큰을 가지고 접근했을 때 공개 API 자체가 401 오류를 반환하는 상황이었다.

나는 **공개 채용공고를 보는 기능은 인증 실패 때문에 중단되지 않아야 한다**고 판단했다.

그래서 인증 실패 중 `UNAUTHENTICATED`와 `TOKEN_EXPIRED`는 익명 사용자로 처리하도록 방향을 정했다.

반면 데이터베이스 장애나 예상하지 못한 서버 오류까지 무조건 무시하지 않도록 했다.

또한 API 문서에 누락됐던 503 응답을 추가하고 공개 조회 API의 401 응답 선언을 정리했다.

여기서 중요한 점은 모든 오류를 예외 처리로 숨기는 것이 아니라, **의도한 인증 실패와 실제 시스템 오류를 구분해야 한다**는 것이다.

---

## 7. CI PASS가 실DB 검증을 의미하지는 않는다

코드 수정 후 GitHub CI 체크가 통과했다.

처음에는 검증이 충분하다고 생각했지만, 선생님께서 중요한 부분을 지적해 주셨다.

> CI PASS만으로는 실DB 검증이라고 볼 수 없다.

확인해 보니 CI의 기본 테스트는 메모리 기반 테스트 대역을 사용하는 구조였다.

PostgreSQL 통합 테스트는 DB 연결 정보가 없으면 skip되도록 설정돼 있었다.

즉, 테스트가 초록색이어도 실제 PostgreSQL에서의 동작까지 검증한 것은 아니었다.

### 별도 테스트 DB 생성

기존 학습용 데이터베이스를 건드리지 않기 위해 PostgreSQL 18에 `career_t18_test`라는 독립적인 데이터베이스를 만들었다.

이후 다음 과정을 진행했다.

1. 역할 및 스키마 bootstrap 적용
2. 마이그레이션 이력 구성
3. 마이그레이션 lint, status, apply 검증
4. Seed 데이터 적용
5. Seed 데이터 검증
6. 실제 PostgreSQL 통합 테스트 및 전체 회귀 테스트 실행

Seed 데이터는 기업 20개, 채용공고 100개로 구성됐다.

### 최종 테스트 결과

| 테스트 | 결과 |
|---|---|
| T18 PostgreSQL 통합 테스트 | **16 passed** |
| 실패 | 0 |
| T18 통합 테스트 skip | 0 |
| 전체 회귀 테스트 | **159 passed** |
| 전체 회귀 skip | 68 |

테스트 실행 후 추가한 테스트 데이터가 남지 않았다는 것도 확인했다.

### 메모리 테스트에서는 발견하지 못한 문제

실DB 테스트 중 기업명 커서 테스트가 실패했다.

원인은 PostgreSQL DB의 `Korean_Korea.949` 콜레이션이었다.

메모리 테스트 대역에서는 Python의 기본 문자열 정렬을 사용했지만, 실제 PostgreSQL은 데이터베이스 콜레이션에 따라 정렬했다.

따라서 영문과 한글의 순서가 테스트에서 가정한 것과 달랐다.

이 문제는 API 구현 오류가 아니라 **테스트 코드가 특정 정렬 순서를 잘못 가정한 문제**였다.

테스트를 수정하면서 고정된 위치를 단언하는 대신 다음과 같은 성질을 검증하도록 변경했다.

- 페이지를 이어 조회했을 때 중복이 없는가?
- 조회 대상이 누락되지 않는가?

이 경험을 통해 테스트 코드는 단순히 예상값과 동일한지 확인하는 것을 넘어, 실제로 보장해야 하는 동작이 무엇인지 검증해야 한다는 사실을 배웠다.

---

## 8. Windows Smart App Control 때문에 pytest가 실행되지 않았다

테스트를 진행하면서 Windows 환경 문제도 만났다.

Python 3.14.7 가상환경에서 pytest를 실행하려고 했는데 `pydantic_core`의 DLL을 불러오지 못했다.

오류 메시지는 다음과 같았다.

`ImportError: DLL load failed while importing _pydantic_core`

처음에는 패키지 설치 문제라고 생각할 수 있는 상황이었다.

하지만 원인을 확인해 보니 Windows Smart App Control의 코드 무결성 정책에 의해 DLL이 차단되고 있었다.

레지스트리의 정책 상태와 Windows Code Integrity 이벤트 로그 ID 3077을 통해 차단 사실을 확인했다.

Smart App Control을 끄는 방법도 있었지만, 비활성화하면 Windows 재설치 전까지 다시 활성화하기 어려운 설정이었다.

따라서 보안 설정을 변경하기보다 다른 해결 방법을 선택했다.

Python 3.12를 사용자 범위에 새로 설치하고 별도 가상환경을 생성했다.

`winget install Python.Python.3.12 --scope user`

이후에는 `pydantic_core`가 정상적으로 로드됐고 pytest도 실행할 수 있었다.

Python 버전에 따른 파일 평판 차이가 원인일 가능성은 있지만, 정확한 차단 이유까지 확인한 것은 아니다.

**배운 점:** 개발 환경에 문제가 생겼을 때 보안 정책부터 해제하기보다, 되돌리기 어려운 변경을 피하면서 안전한 대체 방법을 먼저 찾는 것이 좋다.

---

## 9. GitHub PR 생성과 AI 협업에서의 실수

오늘 구현한 기능은 GitHub Draft PR #128로 정리했다.

Claude Code에 커밋, 최신 main 기준 rebase, push, PR 생성을 맡겼다.

Rebase는 충돌 없이 완료됐고 최종적으로 CI 체크 4개도 모두 통과했다.

다만 AI가 작성한 결과물을 검토하는 과정에서 문제가 발견됐다.

Claude Code가 원래 준비돼 있던 PR 설명 대신 자신의 작업 실행 보고를 PR 본문에 넣었던 것이다.

코드가 정상적으로 작성됐더라도 PR의 내용까지 올바른 것은 아니었다.

본문을 다시 수정하고 검토 결과와 증빙을 정리했다.

또한 PR 생성 명령이 본문 파일 문제로 멈추는 상황도 있었고, 이를 수정해 다시 실행했다.

오늘 AI 협업에서 느낀 점은 분명했다.

**AI에게 작업을 위임할 수는 있지만, 결과의 정확성과 최종 판단까지 위임해서는 안 된다.**

특히 팀 프로젝트에서는 코드뿐 아니라 PR 설명, 테스트 증빙, 이슈 상태까지 모두 협업의 결과물이기 때문이다.

---

## 10. 오늘의 작업 결과와 남은 과제

현재 T18 작업은 구현과 자체 검증을 마친 상태다.

- GitHub Draft PR #128 생성
- 커밋 3개 반영
- 최신 main 기준 rebase 완료
- CI 4개 체크 통과
- PostgreSQL 통합 테스트 16건 통과
- 전체 회귀 테스트 159건 통과
- PR 본문 및 댓글 갱신 완료

다만 **PR 생성과 CI 통과를 프로젝트 완료로 판단하지는 않았다.**

프로젝트에서는 PR 병합, 자동 배포, Staging QA, 일감 종료가 별도의 단계로 구분돼 있다.

따라서 현재 적절한 Redmine 상태는 `In Progress`, 진척률 90%다.

이후 진행할 작업은 다음과 같다.

- 선생님 코드 리뷰 및 병합 판단 확인
- 병합 후 Staging 환경의 실DB 검증
- Redmine 진척률 및 PR 링크 반